# Terra_vault — TrOCR Fine-Tuning Notebook
**GPU-Accelerated Training on Google Colab (T4 / A100)**

### What this notebook does:
1. Installs all dependencies
2. Uploads your `trocr_train.zip` dataset from your PC
3. Fine-tunes `microsoft/trocr-base-stage1` on your land deed crops
4. Evaluates with CER (Character Error Rate) — target < 5%
5. Downloads the trained weights as a zip

### Before running:
- Runtime → Change runtime type → **T4 GPU** → Save
- Run `scripts/prepare_training_data.py` on your PC first
- Have `trocr_train.zip` ready to upload

---

## ✅ Cell 1 — Check GPU

In [ ]:
# Must show a GPU — if not, go to Runtime → Change runtime type → T4 GPU
!nvidia-smi

import torch
print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    raise RuntimeError("❌ No GPU found. Go to Runtime → Change runtime type → T4 GPU")

## 📦 Cell 2 — Install Dependencies

In [ ]:
# Install exact versions tested with Terra_vault's TrOCR stack
!pip install -q \
    transformers==4.40.0 \
    datasets==2.19.0 \
    evaluate==0.4.2 \
    jiwer==3.0.3 \
    pillow \
    tqdm \
    accelerate

print("✅ All packages installed")

## 📂 Cell 3 — Upload Dataset

In [ ]:
import os, zipfile
from google.colab import files

# ── Option A: Upload from your PC (a file picker will appear) ──────────────
print("Uploading trocr_train.zip from your PC...")
print("(This may take a few minutes depending on your internet speed)")
uploaded = files.upload()
zip_path = list(uploaded.keys())[0]

# ── Option B: If you already uploaded to Google Drive, comment out A and use: ──
# from google.colab import drive
# drive.mount('/content/drive')
# zip_path = "/content/drive/MyDrive/trocr_train.zip"

# Extract dataset
os.makedirs("/content/data/trocr_train/images", exist_ok=True)
os.makedirs("/content/data/synthetic_degraded", exist_ok=True)

print(f"\nExtracting {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("/content/data/")

# Count dataset
import json
labels_candidates = [
    "/content/data/trocr_train/labels.jsonl",
    "/content/data/labels.jsonl",
]
labels_path = next((p for p in labels_candidates if os.path.exists(p)), None)
if labels_path is None:
    # Walk to find it
    for root, dirs, fnames in os.walk("/content/data"):
        for fname in fnames:
            if fname == "labels.jsonl":
                labels_path = os.path.join(root, fname)
                break

if labels_path:
    with open(labels_path) as f:
        all_samples = [json.loads(l) for l in f if l.strip()]
    print(f"\n✅ Dataset ready: {len(all_samples)} samples")
    print(f"   Labels file: {labels_path}")
    print("\nFirst 3 samples:")
    for s in all_samples[:3]:
        print(f"  {s}")
else:
    raise FileNotFoundError("labels.jsonl not found in zip. Check your zip structure.")

## 🤖 Cell 4 — Load Base TrOCR Model

In [ ]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

MODEL_NAME = "microsoft/trocr-base-stage1"
print(f"Loading {MODEL_NAME}...")

processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# ⚠️ CRITICAL — These 3 lines prevent training crashes
# Without them, the decoder won't know where to start/stop generating
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# Additional generation config
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_new_tokens = 64

device = torch.device("cuda")
model.to(device)

print(f"\n✅ Model loaded on: {device}")
print(f"   decoder_start_token_id : {model.config.decoder_start_token_id}")
print(f"   pad_token_id           : {model.config.pad_token_id}")
print(f"   eos_token_id           : {model.config.eos_token_id}")
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"   Model parameters       : {total_params:.1f}M")

## 📊 Cell 5 — Build Dataset & Splits

In [ ]:
import os, json, random, torch
from torch.utils.data import Dataset
from PIL import Image


class LandDeedDataset(Dataset):
    """
    Dataset for Terra_vault TrOCR fine-tuning.
    Each sample: image crop (384×64) → ground truth text string.
    """

    def __init__(self, samples, images_base_dir, processor, max_target_length=64):
        self.samples = samples
        self.images_base_dir = images_base_dir
        self.processor = processor
        self.max_len = max_target_length
        self._bad = 0

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img_path = os.path.join(self.images_base_dir, sample["file"])

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            # Return a blank image if crop is missing
            image = Image.new("RGB", (384, 64), color=(255, 255, 255))
            self._bad += 1

        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()

        labels = self.processor.tokenizer(
            sample["text"],
            padding="max_length",
            max_length=self.max_len,
            truncation=True,
        ).input_ids

        # Replace pad token id with -100 so CrossEntropyLoss ignores padding
        labels = [
            -100 if l == self.processor.tokenizer.pad_token_id else l
            for l in labels
        ]

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(labels, dtype=torch.long),
        }


# ── Build splits ──────────────────────────────────────────────────────────────
random.seed(42)
random.shuffle(all_samples)

split = int(0.85 * len(all_samples))    # 85% train / 15% eval
train_samples = all_samples[:split]
eval_samples  = all_samples[split:]

images_base = os.path.dirname(labels_path)   # same folder as labels.jsonl

train_dataset = LandDeedDataset(train_samples, images_base, processor)
eval_dataset  = LandDeedDataset(eval_samples,  images_base, processor)

print(f"Train: {len(train_dataset)} samples")
print(f"   Eval : {len(eval_dataset)} samples")
print(f"   Image base dir: {images_base}")

# Quick sanity check — load one sample
sample_batch = train_dataset[0]
print(f"\n   Sample pixel_values shape : {sample_batch['pixel_values'].shape}")
print(f"   Sample labels shape       : {sample_batch['labels'].shape}")
print(f"   Sample text (decoded)     : {train_samples[0]['text']}")

## 🚀 Cell 6 — Train the Model

**Training takes ~1-2 hours on T4 GPU.**  
Watch the `eval_cer` column — it should go DOWN each epoch.  
Stop when it's below **0.05** (5% character error rate).

In [ ]:
import numpy as np
from jiwer import cer
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
    EarlyStoppingCallback,
)


def compute_metrics(pred):
    """Computes Character Error Rate (CER) on the eval set."""
    label_ids = pred.label_ids.copy()
    pred_ids  = pred.predictions

    # Clip pred_ids to valid vocab range before decoding
    vocab_size = processor.tokenizer.vocab_size
    pred_ids = np.clip(pred_ids, 0, vocab_size - 1)

    pred_str  = processor.batch_decode(pred_ids, skip_special_tokens=True)

    # Replace -100 (padding) with pad_token_id before decoding labels
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    # Filter out empty pairs to avoid jiwer divide-by-zero
    pairs = [(r, h) for r, h in zip(label_str, pred_str) if r.strip()]
    if not pairs:
        return {"cer": 1.0}
    refs  = [p[0] for p in pairs]
    hyps  = [p[1] for p in pairs]

    error_rate = cer(refs, hyps)
    return {"cer": round(float(error_rate), 4)}


# ── Training arguments ────────────────────────────────────────────────────────
# Checkpoints go to Google Drive to survive Colab disconnects
CHECKPOINT_DIR = "/content/drive/MyDrive/terravault_checkpoints"
if not os.path.exists("/content/drive"):
    CHECKPOINT_DIR = "/content/checkpoints"   # fallback if Drive not mounted
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir                  = CHECKPOINT_DIR,
    predict_with_generate       = True,
    generation_max_length       = 64,

    # Eval & save
    evaluation_strategy         = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "cer",
    greater_is_better           = False,     # lower CER = better

    # Training hyperparameters
    num_train_epochs            = 20,
    per_device_train_batch_size = 8,         # reduce to 4 if OOM on T4
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 2,         # effective batch = 16
    learning_rate               = 5e-5,
    weight_decay                = 0.01,
    warmup_steps                = 100,
    lr_scheduler_type           = "cosine",
    fp16                        = True,      # GPU half-precision (2× faster)

    # Logging
    logging_dir                 = "/content/logs",
    logging_steps               = 20,
    report_to                   = "none",
    dataloader_num_workers      = 2,
)

trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = eval_dataset,
    compute_metrics = compute_metrics,
    data_collator   = default_data_collator,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print("🚀 Starting TrOCR fine-tuning...")
print(f"   Checkpoints → {CHECKPOINT_DIR}")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size} × gradient_accum {training_args.gradient_accumulation_steps} = effective {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Stop early if CER doesn't improve for 3 epochs")
print()

trainer.train()

print("\n✅ Training complete!")
print(f"   Best CER: {trainer.state.best_metric:.4f}")

## 📈 Cell 7 — Evaluate on Sample Predictions

In [ ]:
# Visual spot-check: see what the model predicts vs ground truth
import random
from PIL import Image as PILImage
import matplotlib.pyplot as plt
import matplotlib

model.eval()

n_samples = min(6, len(eval_dataset))
indices = random.sample(range(len(eval_dataset)), n_samples)

fig, axes = plt.subplots(n_samples, 1, figsize=(12, n_samples * 2))
if n_samples == 1:
    axes = [axes]

for ax, idx in zip(axes, indices):
    sample = eval_dataset[idx]
    truth  = eval_samples[idx]["text"]

    # Run inference
    pixel_values = sample["pixel_values"].unsqueeze(0).to(device)
    with torch.no_grad():
        generated_ids = model.generate(pixel_values, max_new_tokens=64)
    pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Show crop
    img_path = os.path.join(images_base, eval_samples[idx]["file"])
    try:
        img = PILImage.open(img_path)
        ax.imshow(img, cmap="gray" if img.mode == "L" else None, aspect="auto")
    except:
        ax.set_facecolor("lightgray")

    color = "green" if pred_text.strip() == truth.strip() else "red"
    ax.set_title(
        f"Truth: {truth}\nPred : {pred_text}",
        color=color, fontsize=9, loc="left"
    )
    ax.axis("off")

plt.tight_layout()
plt.savefig("/content/predictions_sample.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → /content/predictions_sample.png")

## 💾 Cell 8 — Save & Download Weights

In [ ]:
import os, zipfile
from google.colab import files

EXPORT_DIR = "/content/trocr_land_deed"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Save the best model
model.save_pretrained(EXPORT_DIR)
processor.save_pretrained(EXPORT_DIR)

print("Files saved:")
total_mb = 0
for f in sorted(os.listdir(EXPORT_DIR)):
    size_mb = os.path.getsize(f"{EXPORT_DIR}/{f}") / 1e6
    total_mb += size_mb
    print(f"  {f:45s}  {size_mb:7.1f} MB")
print(f"  {'TOTAL':45s}  {total_mb:7.1f} MB")

# Zip
zip_out = "/content/trocr_land_deed_finetuned.zip"
with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(EXPORT_DIR):
        zf.write(f"{EXPORT_DIR}/{fname}", fname)

zip_mb = os.path.getsize(zip_out) / 1e6
print(f"\n✅ Zipped → {zip_out}  ({zip_mb:.0f} MB)")
print("\nDownloading to your PC (this may take a few minutes)...")
files.download(zip_out)

## 🏠 Cell 9 — Deploy Instructions (run on your PC after download)

After the zip downloads:

```bash
# 1. Extract the zip
#    → extracts: model.safetensors, config.json, tokenizer files, etc.

# 2. Copy ALL files into:
#    c:\projects\Terra_vault\ml_models\trocr_land_deed\
#    (overwrite existing model.safetensors)

# 3. Restart backend
uvicorn backend.api.main:app --reload

# 4. Check logs for:
#    trocr.loaded_custom_fine_tuned
#    → Your fine-tuned model is now live!

# 5. Run smoke test
python scripts/smoke_test.py
```

### What improves after fine-tuning:
- **Tamil text recognition** — `வேடசந்தூர்`, `கருப்பையா` etc.
- **Land deed numbers** — Survey No, Patta No, Khasra No
- **Degraded document recovery** — `DegradedDocumentRecovery.recover()` gets better base OCR to work with
- **Ensemble OCR confidence** — `avg_confidence` rises above the 0.55 threshold
